# Exploring the MATH dataset

Goal: get familiar with the [Hendrycks MATH](https://arxiv.org/abs/2103.03874) dataset before building any composition or org-layer code. This notebook only loads, samples, and characterizes the data.

**What we want to know by the end:**
1. How are problems and solutions structured?
2. What does the boxed final answer look like across subjects?
3. Which (subject, level) cells produce *rational-number* answers we can compose?
4. What's the distribution across the 7 subjects × 5 levels grid?

No solving, no API calls, no scaffolding here — pure exploration.

## 0. Setup

In [ ]:
# Install once, then comment out
# %pip install datasets pandas matplotlib sympy

In [ ]:
import re
import random
from collections import Counter
from fractions import Fraction

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

random.seed(0)

## 1. Load the dataset

The original `hendrycks/competition_math` is gated by a DMCA notice. Use one of the public mirrors. If the first fails, try the next.

All three carry the same content (problem / solution / level / type).

In [ ]:
MIRRORS = [
    "EleutherAI/hendrycks_math",   # split per subject
    "lighteval/MATH",              # single split, all subjects
    "qwedsacf/competition_math",   # raw mirror
]

ds = None
for name in MIRRORS:
    try:
        ds = load_dataset(name)
        print(f"loaded {name}: {ds}")
        break
    except Exception as e:
        print(f"failed {name}: {e!r}")

assert ds is not None, "no mirror worked — try a different name on HF Hub"

In [ ]:
# Some mirrors split per subject; flatten to one DataFrame so the rest is mirror-agnostic.
def to_dataframe(ds) -> pd.DataFrame:
    frames = []
    for split_name, split in ds.items():
        df = split.to_pandas()
        df["split"] = split_name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

df = to_dataframe(ds)
print(df.shape)
print(df.columns.tolist())
df.head(2)

## 2. Inspect a few raw examples

The fields are `problem`, `solution`, `level`, `type`. The final answer lives inside the solution as `\boxed{...}`.

In [ ]:
def show(row, max_len=600):
    print(f"[{row['type']} | {row['level']}]")
    print("PROBLEM :", row["problem"][:max_len])
    print("SOLUTION:", row["solution"][:max_len])
    print("-" * 80)

for _, row in df.sample(5, random_state=1).iterrows():
    show(row)

## 3. Distribution across subject × level

Use this to decide which cells to draw leaves from.

In [ ]:
pivot = df.pivot_table(index="type", columns="level", values="problem", aggfunc="count", fill_value=0)
# Order the levels nicely ("Level 1" .. "Level 5")
level_order = sorted([c for c in pivot.columns if isinstance(c, str)],
                     key=lambda s: int(re.search(r"\d+", s).group()))
pivot = pivot[level_order]
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("MATH problems per subject × level")
ax.set_ylabel("count")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

## 4. Extracting the boxed final answer

We need a robust extractor for `\boxed{...}` that handles nested braces.

In [ ]:
def extract_boxed(text: str) -> str | None:
    """Return the contents of the LAST \\boxed{...} in text, with brace-balancing."""
    idx = text.rfind(r"\boxed{")
    if idx == -1:
        return None
    i = idx + len(r"\boxed{")
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
            out.append(c)
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
            out.append(c)
        else:
            out.append(c)
        i += 1
    return "".join(out) if depth == 0 else None

df["boxed"] = df["solution"].apply(extract_boxed)
missing = df["boxed"].isna().sum()
print(f"problems with no boxed answer: {missing} / {len(df)}")

In [ ]:
# Browse some boxed values to see the variety of answer types
for _, row in df.dropna(subset=["boxed"]).sample(15, random_state=2).iterrows():
    print(f"[{row['type']:<24} | {row['level']}] {row['boxed']}")

## 5. Filter to rational-number answers

For composition we want a closed type. Try to parse each boxed value as a `Fraction` (covers integers and `\frac{a}{b}` and signed/decimal forms). Anything else is dropped from the *composable pool* — but stays available for inspection.

In [ ]:
FRAC_RE = re.compile(r"^\s*-?\s*\\d?frac\s*\{\s*(-?\d+)\s*\}\s*\{\s*(-?\d+)\s*\}\s*$")
INT_RE  = re.compile(r"^\s*(-?\d+)\s*$")
DEC_RE  = re.compile(r"^\s*(-?\d+\.\d+)\s*$")

def parse_rational(boxed: str | None) -> Fraction | None:
    if boxed is None:
        return None
    s = boxed.strip().replace(" ", "")
    # \frac{a}{b} or \dfrac{a}{b}, possibly with leading minus
    sign = 1
    if s.startswith("-"):
        sign = -1
        s = s[1:]
    m = re.match(r"^\\d?frac\{(-?\d+)\}\{(-?\d+)\}$", s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        if b == 0:
            return None
        return Fraction(sign * a, b)
    m = INT_RE.match(s)
    if m:
        return Fraction(sign * int(m.group(1)), 1)
    m = DEC_RE.match(s)
    if m:
        return Fraction(str(sign * float(m.group(1)))).limit_denominator(10**9)
    return None

df["rational"] = df["boxed"].apply(parse_rational)
rational_mask = df["rational"].notna()
print(f"rational-answer problems: {rational_mask.sum()} / {len(df)} "
      f"({rational_mask.mean():.1%})")

In [ ]:
# How does the rational-answer fraction break down by subject × level?
rat_pivot = (df.assign(is_rational=rational_mask)
               .pivot_table(index="type", columns="level", values="is_rational",
                            aggfunc="mean", fill_value=0))[level_order]
rat_pivot.style.format("{:.0%}").background_gradient(cmap="Greens")

## 6. The composable pool

Final cell: build the pool we'll actually sample leaves from later. We pick a few promising (subject, level) cells where (a) the rational-answer rate is high and (b) the level is low enough that a small model has a fighting chance per leaf.

In [ ]:
candidate_subjects = ["Prealgebra", "Algebra", "Number Theory", "Counting & Probability"]
candidate_levels = ["Level 1", "Level 2"]

pool = df[rational_mask
          & df["type"].isin(candidate_subjects)
          & df["level"].isin(candidate_levels)].copy()
print(f"composable pool size: {len(pool)}")
pool.groupby(["type", "level"]).size().unstack(fill_value=0)

In [ ]:
# Hand-inspect 5 random pool problems to make sure they look reasonable
for _, row in pool.sample(5, random_state=3).iterrows():
    print(f"[{row['type']} | {row['level']}] answer = {row['rational']}")
    print(row["problem"])
    print("-" * 80)

## Next steps (separate notebook)

- Single-leaf accuracy probe: pass each pool problem to Haiku once, measure pass-rate per (subject, level) cell. Pick cells with ~85–95% accuracy as the final leaf pool.
- Hand-build a depth-2 composition and try the flat-baseline path before any organization layer.
- Generator for trees of depth D, branching K, with `Fraction`-typed aggregation.